# YargıRAG — Uçtan Uca Akademik Pipeline (Colab A100)

**Türkçe Hukuki Bilgi Erişimi: Açık Kıyaslama Kümesi, Alana Uyarlanmış Gömme Modeli ve Kaynak-Temelli Sistem**

Bu defter, makaledeki tüm deneyleri **sızıntısız (leakage-free)** biçimde yeniden üretir:

1. Veri yükleme (15.000 karar, açık kaynak)
2. **TurkLegalBench** — karar bazında train/dev/test bölmesi
3. **legal-e5-tr** — yalnızca eğitim bölmesinden zor olumsuz örneklerle ince ayar
4. 8 modelli baseline (TEST bölmesinde)
5. Hibrit erişim (BM25 + dense, RRF)
6. İstatistiksel anlamlılık (eşlemeli bootstrap)
7. Ayrıştırma + hata analizi
8. Kalıcı indeks (demo) + HuggingFace yayını

> **Sızıntısız tasarım kritiktir:** Bir karara ait tüm sorgu/pasajlar tek bölmede tutulur; model ince ayarda test kararlarını görmez. Bu, data leakage'ı engeller ve sonuçları akademik olarak savunulabilir kılar.

## 0. Kurulum

In [ ]:
# GPU kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
# Bağımlılıklar
!pip install -q sentence-transformers faiss-gpu-cu12 rank_bm25 datasets ijson accelerate bitsandbytes scipy huggingface_hub pandas 2>/dev/null
print('deps OK')

In [ ]:
# === src/ kodunu yükle: 'YargiRAG_src.zip'i sol panele SÜRÜKLE-BIRAK, sonra çalıştır ===
import sys, os, zipfile
if os.path.exists('YargiRAG_src.zip'):
    with zipfile.ZipFile('YargiRAG_src.zip') as z:
        z.extractall('src')
    print('src/ açıldı:', os.listdir('src')[:6], '...')
else:
    raise FileNotFoundError('YargiRAG_src.zip bulunamadı — sol panele sürükle-bırak yap')
sys.path.insert(0, 'src')

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/YargiRAG/artifacts'
else:
    WORK = '/content/artifacts'
os.makedirs(WORK, exist_ok=True)
print('Çalışma klasörü:', WORK)

## 1. Veri Yükleme

Açık kaynak `erdem-erdem/Turkish-Law-Documents-700k-clustered` (Yargıtay + Danıştay, ~702K) kümesinden 15.000 karar örneklenir.

In [ ]:
from hf_loader import load_hf_decisions
N_DECISIONS = 15000
decisions = load_hf_decisions(n=N_DECISIONS)
print(f'{len(decisions)} karar yüklendi')

## 2. TurkLegalBench — Sızıntısız Bölme

**İmza katkı 1.** BEIR-uyumlu kıyaslama kümesi; bölme **karar bazında** yapılır. `assert`'ler bölmeler arası örtüşme olmadığını doğrular. Değerlendirme **test** bölmesinde.

In [ ]:
# === 1) Leakage-free split: karar bazında train/dev/test ===
import random
from build_benchmark import build_silver, save_beir
from baselines import load_beir

SEED = 42
random.seed(SEED)

idx = list(range(len(decisions)))
random.shuffle(idx)

n = len(idx)
n_train = int(0.80 * n)
n_dev = int(0.10 * n)

train_decisions = [decisions[i] for i in idx[:n_train]]
dev_decisions   = [decisions[i] for i in idx[n_train:n_train+n_dev]]
test_decisions  = [decisions[i] for i in idx[n_train+n_dev:]]

for split_name, split_decisions in [
    ("train", train_decisions),
    ("dev", dev_decisions),
    ("test", test_decisions),
]:
    split_dir = f"{WORK}/turklegalbench_{split_name}"
    c, q, r = build_silver(split_decisions, max_queries_per_decision=2, seed=SEED)
    save_beir(c, q, r, split_dir)

train_corpus, train_queries, train_qrels = load_beir(f"{WORK}/turklegalbench_train")
dev_corpus, dev_queries, dev_qrels       = load_beir(f"{WORK}/turklegalbench_dev")
test_corpus, test_queries, test_qrels    = load_beir(f"{WORK}/turklegalbench_test")

assert set(train_corpus).isdisjoint(dev_corpus)
assert set(train_corpus).isdisjoint(test_corpus)
assert set(dev_corpus).isdisjoint(test_corpus)

print({
    "train_docs": len(train_corpus), "train_queries": len(train_queries),
    "dev_docs": len(dev_corpus),     "dev_queries": len(dev_queries),
    "test_docs": len(test_corpus),   "test_queries": len(test_queries),
})
print("Karar bazında split temiz.")

# Bundan sonra evaluation için test split kullanacağız
corpus, queries, qrels = test_corpus, test_queries, test_qrels

## 3. Zor Olumsuz Örnek Madenciliği (yalnızca eğitim)

**İmza katkı 2(a).** BM25 ile "yakın ama yanlış" pasajlardan (anchor, positive, negative) üçlüleri; **yalnızca eğitim bölmesinden** — test sızıntısı yok.

In [ ]:
# === 2) Train-only hard negative mining ===
from hard_negatives import mine_bm25_hard_negatives, to_triplet_examples
from rank_bm25 import BM25Okapi
from retriever import tokenize_tr

train_corpus_ids = list(train_corpus.keys())
train_corpus_texts = [train_corpus[c]["text"] for c in train_corpus_ids]
train_bm25 = BM25Okapi([tokenize_tr(t) for t in train_corpus_texts])

pairs_train = [
    (train_queries[qid]["text"], did)
    for qid, dmap in train_qrels.items()
    for did in dmap.keys()
]

mined_train = mine_bm25_hard_negatives(
    pairs_train, train_corpus_texts, train_corpus_ids, train_bm25, tokenize_tr,
    n_neg=4, top_k=30, skip_top=2
)

train_corpus_map = {did: train_corpus[did]["text"] for did in train_corpus_ids}
triplets_train = to_triplet_examples(mined_train, train_corpus_map)

print(f"pairs_train={len(pairs_train):,}")
print(f"triplets_train={len(triplets_train):,}")

## 4. legal-e5-tr — Alana Uyarlanmış İnce Ayar

**İmza katkı 2(b).** `multilingual-e5-base`, üçlülerle MNRL, 3 dönem, fp16. Çıktı: `legal-e5-tr-splitclean`.

In [ ]:
# === 3) Re-train FT model (split-clean) ===
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader

BASE_MODEL = "intfloat/multilingual-e5-base"
FT_OUT_CLEAN = f"{WORK}/legal-e5-tr-splitclean"

model = SentenceTransformer(BASE_MODEL, device="cuda")
model.max_seq_length = 384

train_examples = [
    InputExample(texts=[t["anchor"], t["positive"], t["negative"]])
    for t in triplets_train
]
loader = DataLoader(train_examples, shuffle=True, batch_size=32)
loss = losses.MultipleNegativesRankingLoss(model)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=3,
    warmup_steps=max(1, int(0.1 * len(loader))),
    show_progress_bar=True,
    output_path=FT_OUT_CLEAN,
    use_amp=True,
)
print("Fine-tune tamam:", FT_OUT_CLEAN)

## 5. Baseline Matrisi (TEST bölmesi)

**İmza katkı 3.** Sekiz model sızıntısız test bölmesinde; bellek güvenliği için tek tek yüklenip boşaltılır (`run_suite`).

In [ ]:
# === 4) Re-evaluate all baselines + FT on test split ===
from baselines import run_suite, evaluate_model, comparison_table
from sentence_transformers import SentenceTransformer
import gc, torch

all_results = {}
per_query_store = {}

MODEL_CONFIGS = [
    {"name": "BM25", "type": "bm25"},
    {"name": "BERTurk (mean-pool)", "type": "mean-pool",
     "model_id": "dbmdz/bert-base-turkish-cased",
     "kwargs": {"query_prefix": "", "passage_prefix": ""}},
    {"name": "BERTurk-Legal (mean-pool)", "type": "mean-pool",
     "model_id": "KocLab-Bilkent/BERTurk-Legal",
     "kwargs": {"query_prefix": "", "passage_prefix": ""}},
    {"name": "mE5-small", "type": "sentence-transformer",
     "model_id": "intfloat/multilingual-e5-small"},
    {"name": "mE5-base", "type": "sentence-transformer",
     "model_id": "intfloat/multilingual-e5-base"},
    {"name": "mE5-large", "type": "sentence-transformer",
     "model_id": "intfloat/multilingual-e5-large"},
    {"name": "BGE-m3", "type": "sentence-transformer",
     "model_id": "BAAI/bge-m3",
     "kwargs": {"query_prefix": "", "passage_prefix": ""}},
]

base_results, base_per_query = run_suite(
    MODEL_CONFIGS, corpus, queries, qrels, tokenize_tr,
    top_k=100, device="cuda", batch_size=64
)
all_results.update(base_results)
per_query_store.update(base_per_query)

ft_model = SentenceTransformer(FT_OUT_CLEAN, device="cuda")
spec = {
    "name": "legal-e5-tr (split-clean + hardneg)",
    "type": "sentence-transformer",
    "model": ft_model,
}
agg_ft, per_q_ft, _ = evaluate_model(spec, corpus, queries, qrels, tokenize_tr, top_k=100)
all_results["legal-e5-tr (split-clean + hardneg)"] = agg_ft
per_query_store["legal-e5-tr (split-clean + hardneg)"] = per_q_ft

del ft_model
gc.collect()
torch.cuda.empty_cache()

print(comparison_table(all_results))
for name in ["BM25", "legal-e5-tr (split-clean + hardneg)"]:
    print(name, len(per_query_store[name]))

## 6. Hibrit Erişim (BM25 + legal-e5-tr, RRF)

Seyrek + yoğun erişim, Karşılıklı Sıralama Füzyonu ($k=60$) ile birleştirilir.

In [ ]:
# === 5) Hybrid: BM25 + legal-e5-tr (RRF) ===
import os, gc
import faiss
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from baselines import compute_metrics, aggregate_metrics, comparison_table
from retriever import tokenize_tr

HYBRID_NAME = "BM25 + legal-e5-tr (RRF)"
RRF_K = 60
TOP_K = 100
FETCH_K = 100
BATCH_SIZE = 64
CACHE_PATH = f"{WORK}/ft_corpus_emb_test.npy"

corpus_ids = list(corpus.keys())
corpus_texts = [corpus[c]["text"] for c in corpus_ids]
qlist = list(queries.keys())

bm25 = BM25Okapi([tokenize_tr(t) for t in corpus_texts])

ft_model = SentenceTransformer(FT_OUT_CLEAN, device="cuda")

if os.path.exists(CACHE_PATH):
    emb_corpus = np.load(CACHE_PATH)
else:
    emb_corpus = ft_model.encode(
        [f"passage: {t}" for t in corpus_texts],
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(CACHE_PATH, emb_corpus)

index = faiss.IndexFlatIP(emb_corpus.shape[1])
index.add(emb_corpus)

emb_query = ft_model.encode(
    [f"query: {queries[qid]['text']}" for qid in qlist],
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
).astype(np.float32)

_, dense_idxs = index.search(emb_query, FETCH_K)

hybrid_ranked = {}
for row_i, qid in enumerate(qlist):
    rrf = {}

    for rank, idx in enumerate(dense_idxs[row_i]):
        did = corpus_ids[idx]
        rrf[did] = rrf.get(did, 0.0) + 1.0 / (RRF_K + rank + 1)

    sparse_scores = bm25.get_scores(tokenize_tr(queries[qid]["text"]))
    sparse_order = np.argsort(sparse_scores)[::-1][:FETCH_K]
    for rank, idx in enumerate(sparse_order):
        did = corpus_ids[idx]
        rrf[did] = rrf.get(did, 0.0) + 1.0 / (RRF_K + rank + 1)

    ranked = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:TOP_K]
    hybrid_ranked[qid] = [did for did, _ in ranked]

hybrid_per_q = [compute_metrics(hybrid_ranked[qid], qrels[qid]) for qid in qrels.keys()]
hybrid_agg = aggregate_metrics(hybrid_per_q)

all_results[HYBRID_NAME] = hybrid_agg
per_query_store[HYBRID_NAME] = hybrid_per_q

print(comparison_table({
    k: all_results[k]
    for k in ["BM25", "legal-e5-tr (split-clean + hardneg)", HYBRID_NAME]
    if k in all_results
}))
print(HYBRID_NAME, len(per_query_store[HYBRID_NAME]))

del ft_model
gc.collect()
torch.cuda.empty_cache()

## 7. İstatistiksel Anlamlılık

**İmza katkı 6.** Eşlemeli bootstrap (10.000) ile %95 güven aralığı ve $p$-değeri.

In [ ]:
# === 6) Significance ===
from stats import format_comparison

for name in ["BM25", "legal-e5-tr (split-clean + hardneg)", HYBRID_NAME]:
    print(name, len(per_query_store[name]))

for base in ["BM25", "legal-e5-tr (split-clean + hardneg)"]:
    print(f"\n### {HYBRID_NAME} vs {base}")
    for metric in ["recall@10", "mrr@10", "ndcg@10", "map"]:
        print(format_comparison(
            HYBRID_NAME, per_query_store[HYBRID_NAME],
            base, per_query_store[base],
            metric
        ))
        print()

## 8. Ayrıştırma: Parça (Chunk) Boyutu

Parça-temelli yoğun erişimde 600–1500 karakter etkisi.

In [ ]:
# === Chunk size ablation (chunked dense retrieval) ===
import gc
import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from chunker import build_chunk_records
from baselines import compute_metrics, aggregate_metrics, comparison_table

def evaluate_chunked_dense(model_dir, decisions_subset, chunk_chars=1200, overlap=150,
                           max_seq_length=384, batch_size=64, fetch_k=200, top_k=100):
    model = SentenceTransformer(model_dir, device="cuda")
    model.max_seq_length = max_seq_length

    chunk_records = build_chunk_records(decisions_subset, max_chars=chunk_chars, overlap=overlap)
    chunk_texts = [r["text"] for r in chunk_records]

    chunk_embs = model.encode(
        [f"passage: {t}" for t in chunk_texts],
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(np.float32)

    index = faiss.IndexFlatIP(chunk_embs.shape[1])
    index.add(chunk_embs)

    qids = list(queries.keys())
    q_emb = model.encode(
        [f"query: {queries[qid]['text']}" for qid in qids],
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype(np.float32)

    _, idxs = index.search(q_emb, fetch_k)

    ranked_results = {}
    for qid, row in zip(qids, idxs):
        ranked_docs, seen = [], set()
        for ci in row:
            did = f"d{chunk_records[ci]['karar_id']}"
            if did in seen:
                continue
            seen.add(did)
            ranked_docs.append(did)
            if len(ranked_docs) >= top_k:
                break
        ranked_results[qid] = ranked_docs

    per_q = [compute_metrics(ranked_results[qid], qrels[qid]) for qid in qids]
    agg = aggregate_metrics(per_q)

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return agg, per_q

chunk_results = {}
for chunk_chars in [600, 900, 1200, 1500]:
    label = f"legal-e5-tr chunk={chunk_chars}"
    agg, per_q = evaluate_chunked_dense(
        FT_OUT_CLEAN, test_decisions, chunk_chars=chunk_chars, max_seq_length=384
    )
    chunk_results[label] = agg
    per_query_store[label] = per_q

print(comparison_table(chunk_results))

## 9. Hata Analizi (BM25 vs legal-e5-tr)

Sorgu türüne göre ilk-10 başarımı; alana uyarlanmış model sözcüksel sorgularda bile BM25'i geçer.

In [ ]:
# === Error analysis: BM25 vs legal-e5-tr ===
import gc
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from baselines import encode_bm25, encode_dense

corpus_ids = list(corpus.keys())
corpus_texts = [corpus[c]["text"] for c in corpus_ids]

bm25_ranked = encode_bm25(corpus_ids, corpus_texts, queries, tokenize_tr, top_k=100)

ft_model = SentenceTransformer(FT_OUT_CLEAN, device="cuda")
dense_ranked = encode_dense(ft_model, corpus_ids, corpus_texts, queries, top_k=100)

del ft_model
gc.collect()
torch.cuda.empty_cache()

def hit_at_k(ranked_ids, rel_map, k=10):
    return int(any(did in rel_map for did in ranked_ids[:k]))

rows = []
for qid, rel_map in qrels.items():
    bm25_hit = hit_at_k(bm25_ranked[qid], rel_map, k=10)
    dense_hit = hit_at_k(dense_ranked[qid], rel_map, k=10)

    if dense_hit > bm25_hit:
        winner = "dense"
    elif bm25_hit > dense_hit:
        winner = "bm25"
    else:
        winner = "tie"

    rows.append({
        "qid": qid,
        "query": queries[qid]["text"],
        "type": queries[qid].get("type", "unknown"),
        "winner@10": winner,
        "bm25_hit@10": bm25_hit,
        "dense_hit@10": dense_hit,
        "relevant_ids": list(rel_map.keys())[:5],
        "bm25_top5": bm25_ranked[qid][:5],
        "dense_top5": dense_ranked[qid][:5],
    })

err_df = pd.DataFrame(rows)

display(
    err_df.groupby(["type", "winner@10"]).size()
          .unstack(fill_value=0)
          .assign(total=lambda x: x.sum(axis=1))
          .sort_values("total", ascending=False)
)

print("\nDense wins:")
display(err_df[err_df["winner@10"] == "dense"][[
    "qid", "query", "type", "relevant_ids", "bm25_top5", "dense_top5"
]].head(15))

print("\nBM25 wins:")
display(err_df[err_df["winner@10"] == "bm25"][[
    "qid", "query", "type", "relevant_ids", "bm25_top5", "dense_top5"
]].head(15))

## 10. Kalıcı Erişim İndeksi (Demo)

Tüm corpus için FAISS indeksi → Drive. Yerel `app_pro.py` avukat arayüzü bunu kullanır.

In [ ]:
# === Helper: split-clean model ile index build et ve Drive'a kaydet ===
from chunker import build_chunk_records
from embedder import load_model, embed_passages, build_faiss_index, save_index
import gc, torch, os

def build_ft_index(decisions_subset, out_dir, model_dir, max_chars=1200, overlap=150, batch_size=32):
    print(f"Model: {model_dir}")
    print(f"Output: {out_dir}")
    print(f"Karar sayısı: {len(decisions_subset):,}")

    chunk_records = build_chunk_records(decisions_subset, max_chars=max_chars, overlap=overlap)
    print(f"Chunk sayısı: {len(chunk_records):,}")

    model = load_model(model_dir, device="cuda")
    texts = [r["text"] for r in chunk_records]

    embeddings = embed_passages(model, texts, batch_size=batch_size)
    index = build_faiss_index(embeddings)
    save_index(index, embeddings, chunk_records, out_dir)

    del model, embeddings, index
    gc.collect()
    torch.cuda.empty_cache()

    print("Bitti:", out_dir)
    print("Var mı?", os.path.exists(out_dir))

In [ ]:
# === Tam sürüm: tüm corpus için kalıcı index ===
INDEX_FULL = f"{WORK}/index_ft_splitclean_full"
build_ft_index(
    decisions_subset=decisions,
    out_dir=INDEX_FULL,
    model_dir=FT_OUT_CLEAN,
    max_chars=1200,
    overlap=150,
    batch_size=32,
)

## 11. HuggingFace Yayını

Model + kıyaslama kümesi kamuya açık. Önce HF token ile giriş (Colab secret: `HF_TOKEN`).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from hf_push import push_model, push_benchmark
HF_USER = 'KULLANICI_ADIN'   # <-- HuggingFace kullanıcı adınız
push_benchmark(f'{WORK}/turklegalbench_test', f'{HF_USER}/TurkLegalBench')
push_model(FT_OUT_CLEAN, f'{HF_USER}/legal-e5-tr',
           base_model='intfloat/multilingual-e5-base',
           benchmark_url=f'https://huggingface.co/datasets/{HF_USER}/TurkLegalBench')

## 12. Sonuçları Kaydet (makale tabloları)

In [ ]:
import json
out = {'baseline': all_results, 'chunk_ablation': chunk_results}
with open(f'{WORK}/all_results_final.json', 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', f'{WORK}/all_results_final.json')
print(comparison_table(all_results))